In [1]:
print("hello")

hello


In [2]:
# for i in range(10):
#     print("gawk",i)
import sys
print(sys.executable)


/Users/priyanshupaul/pracs/pythonprac/data-science-prac/venv/bin/python


In [3]:
import pandas as pd

df = pd.read_csv("./dirty_cafe_sales.csv")


print(df.head())

  Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2            2.0         4.0     Credit Card   
1    TXN_4977031    Cake        4            3.0        12.0            Cash   
2    TXN_4271903  Cookie        4            1.0       ERROR     Credit Card   
3    TXN_7034554   Salad        2            5.0        10.0         UNKNOWN   
4    TXN_3160411  Coffee        2            2.0         4.0  Digital Wallet   

   Location Transaction Date  
0  Takeaway       2023-09-08  
1  In-store       2023-05-16  
2  In-store       2023-07-19  
3   UNKNOWN       2023-04-27  
4  In-store       2023-06-11  


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [5]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [6]:
df[["Item","Quantity"]]

,Item,Quantity
0,Coffee,2
1,Cake,4
2,Cookie,4
3,Salad,2
4,Coffee,2
...,...,...
9995,Coffee,2
9996,NaN,3
9997,Coffee,4
9998,Cookie,3


In [7]:
df.iloc[0]

Transaction ID      TXN_1961373
Item                     Coffee
Quantity                      2
Price Per Unit              2.0
Total Spent                 4.0
Payment Method      Credit Card
Location               Takeaway
Transaction Date     2023-09-08
Name: 0, dtype: str

In [8]:
import numpy as np
df["Total Spent"] = pd.to_numeric(df["Total Spent"],errors="coerce")
df.replace(['UNKNOWN', 'ERROR'], np.nan, inplace=True)
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors="coerce")
df['Quantity']       = pd.to_numeric(df['Quantity'],       errors='coerce')
# Case 1: Total Spent missing
mask_T = df['Total Spent'].isna() & df['Quantity'].notna() & df['Price Per Unit'].notna()
df.loc[mask_T, 'Total Spent'] = df.loc[mask_T, 'Quantity'] * df.loc[mask_T, 'Price Per Unit']

# mask_T = df['calculated_total'].isna() & df['Quantity'].notna() & df['Price Per Unit'].notna()
# df.loc[mask_T, 'calculated_total'] = df.loc[mask_T, 'Quantity'] * df.loc[mask_T, 'Price Per Unit']

# Case 2: Quantity missing
mask_Q = df['Quantity'].isna() & df['Total Spent'].notna() & df['Price Per Unit'].notna()
df.loc[mask_Q, 'Quantity'] = df.loc[mask_Q, 'Total Spent'] / df.loc[mask_Q, 'Price Per Unit']

# Case 3: Price Per Unit missing
mask_P = df['Price Per Unit'].isna() & df['Total Spent'].notna() & df['Quantity'].notna()
df.loc[mask_P, 'Price Per Unit'] = df.loc[mask_P, 'Total Spent'] / df.loc[mask_P, 'Quantity']


# df[df["Total Spent"]>10.0]
df[1000:1020]


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1000,TXN_5348445,Salad,5.0,5.0,25.0,Digital Wallet,In-store,2023-11-15
1001,TXN_8920751,Tea,2.0,1.5,3.0,NaN,NaN,2023-04-20
1002,TXN_7215985,Tea,4.0,1.5,6.0,Credit Card,In-store,2023-08-07
1003,TXN_7448296,Cookie,1.0,1.0,1.0,Credit Card,In-store,2023-11-15
1004,TXN_2625324,Coffee,1.0,2.0,2.0,NaN,NaN,2023-09-02
1005,TXN_3542902,Smoothie,1.0,4.0,4.0,NaN,In-store,2023-06-10
1006,TXN_1494608,Tea,3.0,1.5,4.5,Credit Card,In-store,2023-05-07
1007,TXN_7615147,Sandwich,4.0,4.0,16.0,Cash,In-store,2023-05-19
1008,TXN_7225428,Tea,NaN,NaN,3.0,Credit Card,Takeaway,2023-03-07
1009,TXN_3426892,Salad,4.0,5.0,20.0,Credit Card,NaN,2023-05-23


In [9]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])
df = df.sort_values('Transaction Date').reset_index(drop=True)  # ← KEY STEP
df = df.dropna(subset=['Item', 'Total Spent', 'Quantity'])
df['calculated_total'] = df['Quantity'] * df['Price Per Unit']
df['price_mismatch']   = ~np.isclose(df['Total Spent'], df['calculated_total'], rtol=0.01)
df[1000:1020]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,calculated_total,price_mismatch
1118,TXN_9107676,Sandwich,2.0,4.0,8.0,Cash,NaN,2023-02-12,8.0,False
1119,TXN_4937344,Sandwich,1.0,4.0,4.0,Credit Card,In-store,2023-02-12,4.0,False
1120,TXN_8233101,Salad,2.0,5.0,10.0,Cash,In-store,2023-02-12,10.0,False
1121,TXN_9699083,Smoothie,1.0,4.0,4.0,Cash,In-store,2023-02-12,4.0,False
1123,TXN_7804128,Sandwich,5.0,4.0,20.0,Digital Wallet,In-store,2023-02-12,20.0,False
1124,TXN_7640914,Salad,1.0,5.0,5.0,Credit Card,NaN,2023-02-12,5.0,False
1125,TXN_4423879,Juice,5.0,3.0,15.0,NaN,NaN,2023-02-12,15.0,False
1127,TXN_2586830,Juice,1.0,3.0,3.0,NaN,In-store,2023-02-12,3.0,False
1128,TXN_2713279,Cookie,4.0,1.0,4.0,Digital Wallet,In-store,2023-02-12,4.0,False
1129,TXN_9354349,Juice,1.0,3.0,3.0,Credit Card,NaN,2023-02-12,3.0,False


In [10]:
print("\n=== AFTER CLEANING ===")
print("Shape:", df.shape)
print("\nDate range:", df['Transaction Date'].min(), "→", df['Transaction Date'].max())
print("\nFirst 5 rows by date:\n", df[['Transaction Date', 'Item', 'Quantity', 'Total Spent', 'Location']].head())
print("\nLast 5 rows by date:\n",  df[['Transaction Date', 'Item', 'Quantity', 'Total Spent', 'Location']].tail())
print("\nPrice mismatches found:", df['price_mismatch'].sum())
print("\nItem counts:\n",          df['Item'].value_counts())
print("\nPayment Method counts:\n",df['Payment Method'].value_counts())
print("\nLocation counts:\n",      df['Location'].value_counts())


=== AFTER CLEANING ===
Shape: (8979, 10)

Date range: 2023-01-01 00:00:00 → 2023-12-31 00:00:00

First 5 rows by date:
   Transaction Date      Item  Quantity  Total Spent  Location
0       2023-01-01     Juice       1.0          3.0  Takeaway
1       2023-01-01  Smoothie       2.0          8.0  In-store
2       2023-01-01       Tea       5.0          7.5  Takeaway
3       2023-01-01  Sandwich       5.0         20.0  In-store
4       2023-01-01     Juice       5.0         15.0  Takeaway

Last 5 rows by date:
      Transaction Date      Item  Quantity  Total Spent  Location
9995              NaT      Cake       1.0          3.0  Takeaway
9996              NaT      Cake       1.0          3.0       NaN
9997              NaT     Juice       3.0          9.0  In-store
9998              NaT  Smoothie       3.0         12.0       NaN
9999              NaT      Cake       5.0         15.0       NaN

Price mismatches found: 0

Item counts:
 Item
Juice       1167
Coffee      1158
Salad       1

In [11]:
df['Revenue'] = df['Quantity'] * df['Price Per Unit']
daily_df = df.groupby('Transaction Date')['Revenue'].sum().reset_index()
daily_df.head(100)
daily_df["Revenue"].mean()

np.float64(209.87808219178083)

In [12]:
daily_df['lag1'] = daily_df['Revenue'].shift(1)
daily_df['lag2'] = daily_df['Revenue'].shift(2)
daily_df['lag3'] = daily_df['Revenue'].shift(3)
daily_df['lag7'] = daily_df['Revenue'].shift(7)
daily_df['day_of_week'] = daily_df['Transaction Date'].dt.weekday
daily_df['month'] = daily_df['Transaction Date'].dt.month

print(daily_df['lag1'])

0        NaN
1      164.0
2      166.5
3      179.0
4      236.5
       ...  
360    233.5
361    197.0
362    167.0
363    148.5
364    149.0
Name: lag1, Length: 365, dtype: float64


In [13]:
X = daily_df[['lag1', 'lag2', 'lag3','lag7','day_of_week', 'month']]
y = daily_df['Revenue']

In [14]:
# Add more features for better prediction
daily_df['rolling_mean_7'] = daily_df['Revenue'].rolling(window=7).mean()
daily_df['rolling_std_7'] = daily_df['Revenue'].rolling(window=7).std()
daily_df['diff1'] = daily_df['Revenue'].diff(1)

# Fill NaN values from rolling and diff operations
daily_df = daily_df.bfill()

# Update feature set
X = daily_df[['lag1', 'lag2', 'lag3', 'lag7', 'day_of_week', 'month', 'rolling_mean_7', 'rolling_std_7', 'diff1']]
y = daily_df['Revenue']

# Re-split the data
split = int(len(daily_df) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [15]:
split = int(len(daily_df) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [16]:
from xgboost import XGBRegressor

model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4)
model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_met

In [17]:
preds = model.predict(X_test)

In [18]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, preds)
print("R² score:", r2)

R² score: 0.9165004565575083


In [19]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# Time series cross-validation
tscv = TimeSeriesSplit(n_splits=3)

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.8, 0.9, 1.0]
}

# Grid search with time series CV
grid_search = GridSearchCV(
    XGBRegressor(random_state=42), 
    param_grid, 
    cv=tscv, 
    scoring='r2', 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

# Evaluate on test set
best_preds = best_model.predict(X_test)
best_r2 = r2_score(y_test, best_preds)
print("Test R² with best model:", best_r2)

Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}
Best CV score: 0.8877294724659435
Test R² with best model: 0.9231705821461598


In [20]:
from sklearn.metrics import mean_absolute_error

print(mean_absolute_error(y_test, preds))

9.846661606880083


In [21]:
last = daily_df.iloc[-1]

next_input = last[['lag1','lag2','lag3','lag7','day_of_week','month', 'rolling_mean_7', 'rolling_std_7', 'diff1']].to_frame().T
next_input = next_input.astype(float)

next_day_pred = best_model.predict(next_input)
print("Next day revenue:", next_day_pred[0])

Next day revenue: 167.36989


In [22]:
print("Train R2:", r2_score(y_train, model.predict(X_train)))
print("Test R2:", r2_score(y_test, model.predict(X_test)))

Train R2: 0.9986367302416314
Test R2: 0.9165004565575083
